Mount the Drive


In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

CHUNKS_DIR = "/content/gdrive/MyDrive/ifixit_dump"

Mounted at /content/gdrive


In [ ]:
!ls -la /content/gdrive/MyDrive/ifixit_dump
!head -n 2 /content/gdrive/MyDrive/ifixit_dump/chunks.jsonl

total 635935
-rw------- 1 root root 431323937 Feb 25 22:34 chunks.jsonl
-rw------- 1 root root      8925 Feb 23 03:44 errors.log
drwx------ 2 root root      4096 Feb 25 22:44 faiss_bge_small
-rw------- 1 root root    382253 Feb 23 00:40 guide_ids.txt
-rw------- 1 root root 219090601 Feb 23 00:40 guides.jsonl
drwx------ 2 root root      4096 Feb 23 00:38 raw
-rw------- 1 root root    381622 Feb 23 03:45 seen_ids.txt
{"chunk_id": "eca26941bc5187d1e2983961edb6dbb6", "guide_id": 1, "chunk_index": 0, "title": "PowerBook G3 Wallstreet Keyboard Replacement", "url": "https://www.ifixit.com/Guide/PowerBook+G3+Wallstreet+Keyboard+Replacement/1", "device": "PowerBook G3 Wallstreet", "tools": [], "parts": ["{'text': 'G3 WallStreet Keyboard', 'notes': None, 'type': '', 'quantity': 1, 'url': '/Item/G3_WallStreet_Keyboard', 'thumbnail': None, 'isoptional': False}"], "text": "TITLE: PowerBook G3 Wallstreet Keyboard Replacement DEVICE: PowerBook G3 Wallstreet SUMMARY: Replacing this keyboard is easy an

Fast load (sample) + feature engineering

In [ ]:
import os, numpy as np, pandas as pd
import orjson
from tqdm import tqdm

CHUNKS_DIR = "/content/gdrive/MyDrive/ifixit_dump"
chunks_path = os.path.join(CHUNKS_DIR, "chunks.jsonl")

N = 100000  # sample size for EDA; increase to 165000 later if you want
rows = []
with open(chunks_path, "rb") as f:
    for i, line in enumerate(tqdm(f, total=N)):
        if i >= N: break
        rows.append(orjson.loads(line))

df = pd.DataFrame(rows)
print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())

text_col = "text"  # confirmed from your sample line
df[text_col] = df[text_col].astype(str).fillna("")

df["char_len"] = df[text_col].str.len()
df["word_len"] = df[text_col].str.split().str.len()
df["lines"] = df[text_col].str.count("\n") + 1
df["avg_word_len"] = df[text_col].str.findall(r"\b\w+\b").apply(lambda x: (sum(len(w) for w in x)/len(x)) if len(x) else 0)
df["log_words"] = np.log1p(df["word_len"])
df["log_chars"] = np.log1p(df["char_len"])

df[["word_len","char_len","lines","avg_word_len"]].describe()

100%|██████████| 100000/100000 [00:04<00:00, 20913.75it/s]


Loaded: (100000, 9)
Columns: ['chunk_id', 'guide_id', 'chunk_index', 'title', 'url', 'device', 'tools', 'parts', 'text']


,word_len,char_len,lines,avg_word_len
count,100000.000000,100000.000000,100000.0,100000.000000
mean,167.985050,944.870240,1.0,4.445472
std,69.862217,392.447421,0.0,0.341275
min,1.000000,1.000000,1.0,0.000000
25%,114.000000,649.000000,1.0,4.255605
50%,220.000000,1171.000000,1.0,4.417040
75%,220.000000,1234.000000,1.0,4.595455
max,220.000000,3160.000000,1.0,12.000000


Install Plotly (interactive + 3D)

In [ ]:
!pip -q install plotly
import plotly.express as px

3D Chunk Galaxy (interactive + hover text)

In [ ]:
sample = df.sample(min(len(df), 25000), random_state=42).copy()

fig = px.scatter_3d(
    sample,
    x="log_words",
    y="avg_word_len",
    z="lines",
    color="log_chars",
    hover_data={"title": True, "device": True, "guide_id": True, "chunk_index": True, "word_len": True, "char_len": True},
    title="3D Chunk Galaxy: chunk length + structure"
)

fig.update_traces(marker=dict(size=2, opacity=0.65))
fig.show()

Output hidden; open in https://colab.research.google.com to view.

3D Density Nebula (binned distribution)

In [ ]:
s = df.sample(min(len(df), 80000), random_state=7).copy()

xb = pd.cut(s["log_words"], bins=22)
yb = pd.cut(s["avg_word_len"], bins=22)
zb = pd.cut(s["lines"], bins=22)

cube = (
    pd.DataFrame({"xb": xb, "yb": yb, "zb": zb})
      .value_counts()
      .reset_index(name="count")
)

def mid(iv): return (iv.left + iv.right)/2
cube["x"] = cube["xb"].apply(mid)
cube["y"] = cube["yb"].apply(mid)
cube["z"] = cube["zb"].apply(mid)

fig = px.scatter_3d(
    cube, x="x", y="y", z="z",
    size="count", color="count",
    title="3D Density Nebula: where chunks concentrate"
)
fig.update_traces(marker=dict(opacity=0.75))
fig.show()

“Device Universe” (top devices + chunk volume)

In [ ]:
top_dev = (
    df.groupby("device")
      .size()
      .sort_values(ascending=False)
      .head(25)
      .reset_index(name="chunks")
)

fig = px.treemap(
    top_dev,
    path=["device"],
    values="chunks",
    title="Top Devices by #Chunks (Treemap)"
)
fig.show()

In [ ]:
!ls -la /content/gdrive/MyDrive/ifixit_dump/faiss_bge_small

total 544222
-rw------- 1 root root 254243373 Feb 25 23:21 index.faiss
-rw------- 1 root root 303038878 Feb 25 23:21 meta.jsonl


1) Load meta.jsonl (sample) + inspect columns (fast)

In [ ]:
import os, pandas as pd, orjson
from tqdm import tqdm

META_DIR = "/content/gdrive/MyDrive/ifixit_dump/faiss_bge_small"
meta_path = os.path.join(META_DIR, "meta.jsonl")

M = 80000  # sample; can raise later
mrows = []
with open(meta_path, "rb") as f:
    for i, line in enumerate(tqdm(f, total=M)):
        if i >= M: break
        mrows.append(orjson.loads(line))

mdf = pd.DataFrame(mrows)
print("Meta shape:", mdf.shape)
print("Meta columns:", mdf.columns.tolist())
mdf.head(2)

100%|██████████| 80000/80000 [00:02<00:00, 27647.95it/s]


Meta shape: (80000, 9)
Meta columns: ['chunk_id', 'guide_id', 'chunk_index', 'title', 'url', 'device', 'tools', 'parts', 'preview']


,chunk_id,guide_id,chunk_index,title,url,device,tools,parts,preview
0,eca26941bc5187d1e2983961edb6dbb6,1,0,PowerBook G3 Wallstreet Keyboard Replacement,https://www.ifixit.com/Guide/PowerBook+G3+Wall...,PowerBook G3 Wallstreet,[],"[{'text': 'G3 WallStreet Keyboard', 'notes': N...",TITLE: PowerBook G3 Wallstreet Keyboard Replac...
1,ea66c06c1e1c05fa9f1aa39d98dc5bc1,1,1,PowerBook G3 Wallstreet Keyboard Replacement,https://www.ifixit.com/Guide/PowerBook+G3+Wall...,PowerBook G3 Wallstreet,[],"[{'text': 'G3 WallStreet Keyboard', 'notes': N...",a spudger downward between each plastic strain...


2) Sunburst (Device → Guide Title)

In [14]:
import plotly.graph_objects as go

# Use the same aggregated data 'sun'
top = sun.copy()

# Create node list
devices = top[dev_col].unique().tolist()
guides = top[title_col].unique().tolist()

node_labels = devices + guides
node_index = {lbl:i for i,lbl in enumerate(node_labels)}

sources = top[dev_col].map(node_index).tolist()
targets = top[title_col].map(node_index).tolist()
values  = top["chunks"].tolist()

fig = go.Figure(data=[go.Sankey(
    node=dict(label=node_labels, pad=15, thickness=15),
    link=dict(source=sources, target=targets, value=values)
)])
fig.update_layout(title_text="Sankey: Device → Guide Title (flow size = #chunks)", font_size=10)
fig.show()

In [15]:
emb_candidates = [c for c in mdf.columns if any(k in c.lower() for k in ["emb", "vector"])]
emb_candidates[:20], len(emb_candidates)

([], 0)

In [18]:
!pip -q install plotly orjson tqdm scikit-learn umap-learn

import os, numpy as np, pandas as pd, orjson
from tqdm import tqdm
import plotly.express as px
import plotly.graph_objects as go

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

In [19]:
CHUNKS_DIR = "/content/gdrive/MyDrive/ifixit_dump"
chunks_path = os.path.join(CHUNKS_DIR, "chunks.jsonl")

N = 120000  # increase up to 165k later if you want
rows = []
with open(chunks_path, "rb") as f:
    for i, line in enumerate(tqdm(f, total=N)):
        if i >= N: break
        rows.append(orjson.loads(line))

df = pd.DataFrame(rows)
print("Loaded:", df.shape)
print("Columns:", df.columns.tolist())

df["text"] = df["text"].astype(str).fillna("")
df["word_len"] = df["text"].str.split().str.len()
df["char_len"] = df["text"].str.len()

100%|██████████| 120000/120000 [00:02<00:00, 42148.39it/s]


Loaded: (120000, 9)
Columns: ['chunk_id', 'guide_id', 'chunk_index', 'title', 'url', 'device', 'tools', 'parts', 'text']


In [20]:
sun = (
    df.groupby(["device", "title"])
      .size()
      .reset_index(name="chunks")
      .sort_values("chunks", ascending=False)
)

# keep it readable: top 20 devices + top 20 guides per device
top_devices = sun.groupby("device")["chunks"].sum().sort_values(ascending=False).head(20).index
sun = sun[sun["device"].isin(top_devices)]
sun = sun.groupby("device").head(20)

fig = px.sunburst(
    sun,
    path=["device", "title"],
    values="chunks",
    title="Sunburst: Device → Guide Title (size = #chunks)"
)
fig.show()